# 11 - Ghost Job Classifier

**Goal**: Build a binary classifier that flags probable ghost jobs — postings with no genuine
intent to hire.

**Research Theme 2**: What signals identify ghost jobs?

**The labeling problem**: We don't have ground truth labels for ghost jobs.
We construct heuristic labels from behavioral signals:
- **High views, very low applies**: A ghost job stays visible for a long time but attracts
  almost no applications — likely because experienced applicants can sense something is off.
- **Heuristic**: `views > 200 AND applies < 5` → likely ghost

**Pipeline**:
```
postings_features.parquet
  → filter rows with views + applies data
  → create heuristic ghost label
  → train Logistic Regression baseline → XGBoost
  → evaluate with precision/recall/F1 (class imbalance!)
  → save model
```

**Learning concepts**: imbalanced classification, precision vs recall trade-off,
confusion matrix, heuristic labeling, threshold tuning.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from talentlens.config import (
    GHOST_JOB_MODEL_PATH,
    POSTINGS_FEATURES_PARQUET,
    RANDOM_SEED,
)
from talentlens.plots import save_fig

pd.set_option('display.max_columns', None)
%matplotlib inline

FORCE_RECOMPUTE = False

logger.info("Libraries loaded.")

## 1. Load Data & Create Ghost Job Labels

### Why heuristic labels?

We don't have verified ghost job labels from LinkedIn. Instead, we reason about
what behavioral pattern a ghost job would produce:

1. **High visibility** (many views) — the job is actively promoted/shown
2. **Very few applications** — applicants either can't apply or choose not to
3. **Low engagement ratio** (applies/views < 0.02) — well below the 0.149 dataset median

This is an **unsupervised to supervised** trick: we convert domain knowledge into labels,
then train a model to generalize the pattern to rows where we don't have applies/views data.

In [ ]:
df = pd.read_parquet(POSTINGS_FEATURES_PARQUET)
logger.info(f"Loaded {len(df):,} postings")

# Only rows with both views and applies can be labeled
df_eng = df[df['views'].notna() & df['applies'].notna()].copy()
logger.info(f"Rows with views + applies: {len(df_eng):,}")

# Compute engagement ratio
df_eng['apply_rate'] = df_eng['applies'] / df_eng['views'].clip(lower=1)

# Ghost job heuristic:
# - views > 200: visible enough to have had a chance to get applications
# - applies < 5: essentially no engagement
VIEWS_THRESHOLD = 200
APPLIES_THRESHOLD = 5

df_eng['is_ghost'] = (
    (df_eng['views'] > VIEWS_THRESHOLD) &
    (df_eng['applies'] < APPLIES_THRESHOLD)
).astype(int)

ghost_count = df_eng['is_ghost'].sum()
total = len(df_eng)
print(f"\nGhost job labels:")
print(f"  Ghost:     {ghost_count:,} ({ghost_count/total*100:.1f}%)")
print(f"  Non-ghost: {total - ghost_count:,} ({(total-ghost_count)/total*100:.1f}%)")
print(f"\nClass imbalance ratio: 1:{(total-ghost_count)//max(ghost_count,1)}")
print("\nThis is imbalanced — we need precision/recall metrics, not just accuracy.")

In [ ]:
# Compare ghost vs non-ghost posting characteristics
print("Mean feature values: Ghost vs Non-Ghost")
print("=" * 55)
compare_cols = [
    'views', 'applies', 'apply_rate',
    'desc_word_count', 'sentiment_polarity',
    'senior_signal_count', 'max_years_required', 'n_skills'
]
comparison = df_eng.groupby('is_ghost')[compare_cols].mean().round(2).T
comparison.columns = ['Non-Ghost', 'Ghost']
comparison['Ratio (Ghost/Non)'] = (comparison['Ghost'] / comparison['Non-Ghost'].clip(lower=0.001)).round(2)
print(comparison)

## 2. Feature Matrix

In [ ]:
# Features available for ALL postings (no views/applies — we want to generalize later)
exp_order = {
    'Internship': 0, 'Entry level': 1, 'Associate': 2,
    'Mid-Senior level': 3, 'Director': 4, 'Executive': 5, 'Unknown': 2
}
df_eng['exp_level_encoded'] = df_eng['experience_level'].map(exp_order).fillna(2)
df_eng['is_remote_int'] = df_eng['is_remote'].astype(int)
df_eng['sponsored_int'] = df_eng['sponsored'].fillna(0).astype(int)

FEATURE_COLS = [
    'desc_word_count',
    'sentiment_polarity',
    'sentiment_subjectivity',
    'senior_signal_count',
    'max_years_required',
    'n_skills',
    'exp_level_encoded',
    'is_remote_int',
    'sponsored_int',
]

X = df_eng[FEATURE_COLS].fillna(0).values
y = df_eng['is_ghost'].values

print(f"Feature matrix: {X.shape}")
print(f"Ghost rate in dataset: {y.mean()*100:.1f}%")

## 3. Logistic Regression Baseline

**Handling class imbalance**: Use `class_weight='balanced'` so the model pays more
attention to the rare ghost class. Without this, the model would just predict
"non-ghost" for everything and achieve ~90% accuracy — useless.

**Why precision/recall over accuracy?**
- **Precision**: Of all predicted ghosts, how many are actually ghost jobs? (False positive cost)
- **Recall**: Of all actual ghost jobs, how many did we catch? (False negative cost)
- For this use case, **recall matters more** — we'd rather flag a real job as ghost than miss a ghost.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_SEED,
    ))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_f1 = cross_val_score(lr_pipe, X_train, y_train, cv=skf, scoring='f1')
cv_roc = cross_val_score(lr_pipe, X_train, y_train, cv=skf, scoring='roc_auc')

print(f"Logistic Regression 5-fold CV:")
print(f"  F1:      {cv_f1.mean():.3f} ± {cv_f1.std():.3f}")
print(f"  ROC-AUC: {cv_roc.mean():.3f} ± {cv_roc.std():.3f}")

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_proba_lr = lr_pipe.predict_proba(X_test)[:, 1]

print(f"\nLogistic Regression Test Set:")
print(classification_report(y_test, y_pred_lr, target_names=['Non-Ghost', 'Ghost']))

## 4. XGBoost Classifier

In [ ]:
if not FORCE_RECOMPUTE and GHOST_JOB_MODEL_PATH.exists():
    logger.info(f"Loading cached model from {GHOST_JOB_MODEL_PATH}")
    best_model = joblib.load(GHOST_JOB_MODEL_PATH)
else:
    try:
        from xgboost import XGBClassifier

        # scale_pos_weight handles imbalance in XGBoost
        neg_count = (y_train == 0).sum()
        pos_count = (y_train == 1).sum()
        scale = neg_count / max(pos_count, 1)

        xgb = XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.05,
            scale_pos_weight=scale,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_SEED,
            verbosity=0,
            eval_metric='logloss',
        )

        cv_f1_xgb = cross_val_score(xgb, X_train, y_train, cv=skf, scoring='f1')
        cv_roc_xgb = cross_val_score(xgb, X_train, y_train, cv=skf, scoring='roc_auc')
        logger.info(f"XGBoost CV F1: {cv_f1_xgb.mean():.3f}, ROC-AUC: {cv_roc_xgb.mean():.3f}")

        xgb.fit(X_train, y_train)
        best_model = xgb
    except ImportError:
        logger.warning("XGBoost not installed — using Logistic Regression.")
        best_model = lr_pipe

    GHOST_JOB_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_model, GHOST_JOB_MODEL_PATH)
    logger.info(f"Model saved to {GHOST_JOB_MODEL_PATH}")

y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

print(f"Best Model Test Set:")
print(classification_report(y_test, y_pred_best, target_names=['Non-Ghost', 'Ghost']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_best):.3f}")

## 5. Confusion Matrix & Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['Non-Ghost', 'Ghost'],
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title('Confusion Matrix — Ghost Job Classifier')

# Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_best)
axes[1].plot(recall, precision, color='darkorange', lw=2)
axes[1].set_xlabel('Recall (% of real ghost jobs caught)')
axes[1].set_ylabel('Precision (% of flagged jobs that are ghost)')
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])
# Mark the default 0.5 threshold operating point
default_prec = precision[len(thresholds[thresholds >= 0.5])]
default_rec = recall[len(thresholds[thresholds >= 0.5])]
axes[1].scatter([default_rec], [default_prec], s=80, color='red', zorder=5,
                label=f'Threshold 0.5\n(P={default_prec:.2f}, R={default_rec:.2f})')
axes[1].legend()

plt.tight_layout()
save_fig(fig, 'ghost_job_precision_recall')
plt.show()

## 6. Score the Full Dataset

In [ ]:
# Score all postings (not just those with views/applies)
exp_order = {
    'Internship': 0, 'Entry level': 1, 'Associate': 2,
    'Mid-Senior level': 3, 'Director': 4, 'Executive': 5, 'Unknown': 2
}
df['exp_level_encoded'] = df['experience_level'].map(exp_order).fillna(2)
df['is_remote_int'] = df['is_remote'].astype(int)
df['sponsored_int'] = df['sponsored'].fillna(0).astype(int)

X_all = df[FEATURE_COLS].fillna(0).values
df['ghost_prob'] = best_model.predict_proba(X_all)[:, 1]

# Threshold: flag as ghost if probability > 0.5
df['is_ghost_flagged'] = (df['ghost_prob'] > 0.5).astype(int)

n_flagged = df['is_ghost_flagged'].sum()
print(f"Postings flagged as ghost jobs: {n_flagged:,} ({n_flagged/len(df)*100:.1f}%)")

print("\nTop 10 most likely ghost jobs:")
top_ghosts = df.nlargest(10, 'ghost_prob')[[
    'title', 'company_name', 'location', 'views', 'applies', 'ghost_prob'
]]
print(top_ghosts.to_string(index=False))

## Summary

### What we built
- **Heuristic labels**: `views > 200 AND applies < 5` → probable ghost job
- **Logistic Regression baseline**: interpretable, handles imbalance with `class_weight='balanced'`
- **XGBoost**: better captures non-linear patterns in ghost job signals
- **`ghost_prob` column**: probability score for every posting in the dataset

### Key finding
The model learns that **high-visibility jobs with almost no applications** are the clearest signal.
Description features (word count, sentiment) add a small but real boost.

### Limitations
- Heuristic labels are imperfect — some "ghost" labels may be real jobs with niche audiences
- We can't verify our findings against ground truth
- Duration data (`days_open`) is only available for 1,071 rows — can't use as a feature

---
**→ Next**: Notebook 12 — Entry-Level Paradox Statistical Analysis